# Cours ULB PHYS-F-450 (2026): Notebook 0 - Introduction à qgs


Ce notebook est le premier d'une série d'exercices pour le cours [PHYS-F-450](https://www.ulb.be/fr/programme/phys-f450) donné à l'ULB.

Dans ce notebook, vous apprendrez :

1. **Comment installer le framework [qgs](https://github.com/Climdyn/qgs) pour la modélisation climatique et météorologique d'ordre réduit**
2. **Comment exécuter le modèle**
3. Déterminer les points d'équilibre du modèle et leur stabilité
4. Comment calculer le plus grand exposant de Lyapunov d'un modèle, et le lien avec sa prévisibilité
5. Comment calculer le spectre complet des exposants de Lyapunov

Ce premier notebook sert d'introduction au framework qgs et illustre les tâches 1 et 2 décrites ci-dessus. Son but est de montrer comment utiliser le modèle, de vous permettre de vous familiariser avec ses paramètres, et de finalement l'intégrer pour déterminer ses trajectoires.

## Installation du framework

Si vous n'avez pas encore [installer qgs](https://github.com/jodemaey/cours-ULB-2026/tree/main#installation-requise), vous pouvez le faire directement dans ce notebook, en décommentant la ligne ci-dessous et en la lançant:

In [ ]:
# !pip install qgs

## Faire tourner un modèle

À l'aide de qgs, nous allons définir une atmosphère quasi-géostrophique simple à deux couches, tronquée au nombre d'onde 2 sur un beta plane (plan beta) avec une orographie simple (une montagne et une vallée). Nous intégrerons ensuite ce modèle et tracerons ses trajectoires.

Pour plus de détails sur ce modèle, veuillez consulter les articles suivants :

* Reinhold, B. B., & Pierrehumbert, R. T. (1982). *Dynamics of weather regimes: Quasi-stationary waves and blocking*. Monthly Weather Review, **110** (9), 1105-1145. [doi:10.1175/1520-0493(1982)110%3C1105:DOWRQS%3E2.0.CO;2](https://doi.org/10.1175/1520-0493(1982)110%3C1105:DOWRQS%3E2.0.CO;2)

ou dans la [documentation](https://qgs.readthedocs.io/en/latest/files/model/oro_model.html).

**Pour commencer, vous pouvez simplement exécuter cellule après cellule, jusqu'au bas du notebook.**

### Import des modules

Nous devons d'abord importer les modules nécessaires aux calculs:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

et importer les modules requis de qgs:

In [ ]:
from qgs.params.params import QgParams
from qgs.integrators.integrator import RungeKuttaIntegrator, RungeKuttaTglsIntegrator
from qgs.functions.tendencies import create_tendencies
from qgs.plotting.util import std_plot

et les "diagnostics":

In [ ]:
from qgs.diagnostics.streamfunctions import MiddleAtmosphericStreamfunctionDiagnostic
from qgs.diagnostics.variables import VariablesDiagnostic
from qgs.diagnostics.multi import MultiDiagnostic

### Définition du modèle

D'abord, définissons quelques paramètres généraux:

In [ ]:
# Pas de temps d'intégration
dt = 0.1
# On veut sauver l'état du modèle tous les 5 pas de temps
write_steps = 5
# Durée pendant laquelle on intègre un transient initial 
# du modèle (en unité de temps du modèle)
transient_time = 200000.
# Temps d'intégration des trajectoires du modèle
integration_time = 100000.

Nous allons maintenant créer l'objet des paramètres du modèle. Cet objet définit entièrement la version et les paramètres du modèle. Il sera utilisé par qgs pour construire les tendances $\boldsymbol{f}$ du modèle.

In [ ]:
model_parameters = QgParams()

Nous pouvons maintenant configurer certains paramètres avec la méthode [`set_params`](https://qgs.readthedocs.io/en/latest/files/technical/configuration.html#qgs.params.params.Params.set_params) de l'objet [`QgParams`](https://qgs.readthedocs.io/en/latest/files/technical/configuration.html#qgs.params.params.QgParams) :

In [ ]:
# nous définissons ici la latitude à 50 degrés et une amplitude prédéfinie du gradient de température méridien
model_parameters.set_params({'phi0_npi': np.deg2rad(50.)/np.pi, 'hd':0.045})

et indiquer que nous souhaitons un canal atmosphérique pour l'atmosphère, avec des modes de Fourier jusqu'au nombre d'onde 2 dans chaque direction spatiale ($x$ et $y$ seront précisés ci-dessous) :

In [ ]:
model_parameters.set_atmospheric_channel_fourier_modes(2, 2)

Nous définissons également une certaine orographie :

In [ ]:
model_parameters.ground_params.set_orography(0.2, 1)

et indiquons l'amplitude du gradient de la température méridionale d'équilibre qui "force" le modèle :

In [ ]:
model_parameters.atemperature_params.set_thetas(0.1, 0)

le `0` indiquant ici qu'on force suivant le premier mode $\sim \cos(y)$ où $y \in [0, \pi]$ est la "latitude" du modèle (voir plus bas). On a donc une température d'équilibre plus élevée au sud qu'au nord du domaine.

La configuration du modèle est terminée. Nous pouvons maintenant afficher les paramètres pour vérifier notre travail :

In [ ]:
model_parameters.print_params()

Nous sommes maintenant prêts à fournir l'objet contenant les paramètres à qgs pour obtenir les tendances $\boldsymbol{f}$ qui nous permettront d'intégrer l'équation différentielle $\dot{\boldsymbol{x}} = \boldsymbol{f}(\boldsymbol{x})$.

In [ ]:
%%time
f, Df = create_tendencies(model_parameters)

Deux objets sont en réalité retournés, les tendances $\boldsymbol{f}$ mais aussi son Jacobien $\boldsymbol{Df}$, qui sera utile plus tard pour calculer les exposants de Lyapunov.

### Intégration temporelle (trajectoires du modèle)

Nous pouvons maintenant intégrer notre modèle à l'aide d'un intégrateur, un algorithme qui nous fournira la solution de l'équation différentielle ci-dessus. Il existe de nombreuses options pour ce faire (voir plus bas), mais nous utiliserons ici l'intégrateur [`RungeKuttaIntegrator`](https://qgs.readthedocs.io/en/latest/files/technical/integrators.html#qgs.integrators.integrator.RungeKuttaIntegrator) intégré à qgs (un intégrateur de [Runge-Kutta d'ordre 4](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods)) :

In [ ]:
integrator = RungeKuttaIntegrator()

Nous demanderons à cet intégrateur d'utiliser notre modèle nouvellement défini:

In [ ]:
integrator.set_func(f)

Nous pouvons maintenant partir d'une petite condition initiale aléatoire et intégrer sur une période transitoire pour obtenir une condition initiale sur les attracteurs.

In [ ]:
%%time
ic = np.random.rand(model_parameters.ndim)*0.1
integrator.integrate(0., transient_time, dt, ic=ic, write_steps=write_steps)  # avec write_steps=0, l'intégrateur gardera seulement le point final en mémoire
time, transient = integrator.get_trajectories()

(voir les options et arguments de la fonction [`integrate`](https://qgs.readthedocs.io/en/latest/files/technical/integrators.html#qgs.integrators.integrator.RungeKuttaIntegrator.integrate) de l'intégrateur, qui lui demande ici d'effectuer l'intégration temporelle).

Il n'est pas inintéressant de tracer l'évolution temporelle cette première trajectoire transitoire, par exemple celle de la première variable: 

In [ ]:
var = 0
plt.figure(figsize=(10, 8))

plt.plot(model_parameters.dimensional_time*time, transient[var])

plt.xlabel('time (days)')
plt.ylabel('$x_1$');

et l'on voit qu'en fait le modèle converge assez rapidement vers son attracteur, dû au caractère *dissipatif* du modèle. Noter aussi qu'on a converti les unités temporelles du modèle en jours via un facteur de conversion. On peut donc utiliser le dernier point de la trajectoire comme condition initiale pour obtenir une trajectoire entièrement sur celui-ci:

In [ ]:
ic = transient[:, -1]

Maintenant que nous avons obtenu une condition initiale sur l'attracteur, nous pouvons donc intégrer davantage pour obtenir une trajectoire sur l'attracteur.

In [ ]:
%%time
integrator.integrate(0., integration_time, dt, ic=ic, write_steps=write_steps)
time, trajectory = integrator.get_trajectories()

### Tracé de la trajectoire du modèle

L'évolution temporelle typique d'une variable du modèle en fonction du temps peut maintenant être représentée graphiquement, par exemple pour la deuxième variable :

In [ ]:
var = 1
plt.figure(figsize=(10, 8))

plt.plot(model_parameters.dimensional_time*time, trajectory[var])

plt.xlabel('time (days)')
plt.ylabel('$x_2$');

On peut également représenter graphiquement, par exemple, les deuxième et troisième variables dans un plan.

In [ ]:
varx = 2
vary = 1
plt.figure(figsize=(10, 8))

plt.plot(trajectory[varx], trajectory[vary], marker='o', ms=0.07, ls='')

plt.xlabel('$x_3$')
plt.ylabel('$x_2$');

Ceci montre une projection bidimensionnelle de l'[attracteur](https://en.wikipedia.org/wiki/Attractor) du modèle, c'est-à-dire l'ensemble des états vers lesquels le modèle tend à évoluer, pour une grande variété de conditions initiales du système. On voit donc une section planaire de l'attracteur résidant ici dans un espace à 20 dimensions

Chaque point représente un état du modèle. $x_2 = \psi_{\rm{a},2}$ et $x_3 = \psi_{\rm{a},3}$ correspondent aux deuxième et troisième variables du modèle. Ce sont les coefficients *spectraux* du modèle. En effet, qgs est un modèle spectral, ce qui signifie que tous ses champs sont développés en modes spatiaux. Ici, par exemple, on a deux champs : la fonction de courant barotrope à 500 hPa $\psi_{\rm{a}}$ et la fonction de courant barocline à 500 hPa (reliée à la température) $\theta_{\rm{a}}$. Ces expressions sont développées sur les 10 modes (car nous avons décidé de les étendre jusqu'au nombre d'onde 2 dans chaque direction) :

$$\psi_{\rm{a}} = \sum_{i=1}^{10} \, \psi_{\rm{a},i} \, F_i(x,y)\qquad\qquad \mathrm{(1)}$$
$$\theta_{\rm{a}} = \sum_{i=1}^{10} \, \theta_{\rm{a},i} \, F_i(x,y) \qquad\qquad\,\, \mathrm{(2)}$$

où :

$$ F_1(x,y) = \sqrt{2}\, \cos(y), \\ F_2(x,y) = 2\, \cos(n x)\, \sin(y), \\ F_3(x,y) = 2\, \sin(n x)\, \sin(y), \\ F_4(x,y) = \sqrt{2}\, \cos(2y), \\ F_5(x,y) = 2 \cos(n x) \sin(2y), \\ F_6(x,y) = 2 \sin(n x) \sin(2y), \\ \vdots $$

où $x$ et $y$ sont les coordonnées adimensionnelles sur le domaine spatial du modèle (un plan $\beta$) : $x \in [0, 2\pi/n]$ et $y \in [0,\pi]$, avec $n$ le rapport d'aspect du domaine, c'est-à-dire le rapport entre son étendue zonale et méridienne.

Voici les 10 modes (illustrés ici pour un rapport d'aspect $n=1$) sur lesquels les champs spatiaux $\psi_{\rm{a}}$ et $\theta_{\rm{a}}$ du modèle sont décomposés :

![Les 10 premières fonctions de base](https://raw.githubusercontent.com/Climdyn/qgs/master/documentation/source/files/model/figures/visualise_basisfunctions_atmosphere.png)

En intégrant $\dot{\boldsymbol{x}} = \boldsymbol{f}(\boldsymbol{x})$, on intègre une équation différentielle décrivant l'évolution temporelle des coefficients spectraux $\psi_{\rm{a}, i}$ et $\theta_{\rm{a}, i}$. Dans notre cas, $\boldsymbol{x} = (\psi_{\rm{a}, 1}, \ldots, \psi_{\rm{a}, 10}, \theta_{\rm{a}, 1}, \ldots, \theta_{\rm{a}, 10})$, et la résolution de l'équation différentielle du modèle nous permet de déterminer $\boldsymbol{x}(t)$. À partir des équations (1) et (2), il est alors aisé de reconstituer l'évolution temporelle $\psi_{\rm{a}}(t)$ et $\theta_{\rm{a}}(t)$ des champs spatiaux du modèle. C'est précisément le rôle des objets [`Diagnostic`](https://qgs.readthedocs.io/en/latest/files/technical/diagnostics.html) du modèle, et ce sera l'objet de la section suivante.

> **Info :** Consultez cette [page](https://qgs.readthedocs.io/en/latest/files/model/oro_model.html) pour plus de détails sur l'expansion spectrale.

### Calcul et représentation graphique des champs spatiaux du modèle

Le calcul et la représentation graphique des champs du modèle peuvent être effectués à l'aide d'objets Python appelés diagnostics (et décrits [ici](https://qgs.readthedocs.io/en/latest/files/technical/diagnostics.html)), qui prennent la sortie de l'intégration du modèle (c'est-à-dire les coefficients spectraux) et la convertissent en champs spatiaux. Une liste des diagnostics actuellement disponibles est incluse dans cette [section](https://qgs.readthedocs.io/en/latest/files/user_guide.html#analyzing-the-model-s-output-with-diagnostics) du guide de l'utilisateur.

Pour ce tutoriel, nous allons calculer le champ spatial de la hauteur géopotentielle à 500 hPa. Ceci peut être réalisé avec la fonction [`MiddleAtmosphericStreamfunctionDiagnostic`](https://qgs.readthedocs.io/en/latest/files/technical/diagnostics.html#qgs.diagnostics.streamfunctions.MiddleAtmosphericStreamfunctionDiagnostic) qui calcule $\psi_{\rm a}(x, y)$.

Pour le calculer, nous commençons par instancier l'objet :

In [ ]:
psi = MiddleAtmosphericStreamfunctionDiagnostic(model_parameters, geopotential=True)

L'instruction `geopotential=True` indique simplement que nous souhaitons obtenir la hauteur géopotentielle. Notez également qu'il est nécessaire de transmettre l'objet `model_parameters` défini précédemment.

Nous devons maintenant transmettre les données d'intégration du modèle à l'aide de la méthode [`set_data`](https://qgs.readthedocs.io/en/latest/files/technical/diagnostics.html#qgs.diagnostics.base.Diagnostic.set_data) :

In [ ]:
psi.set_data(time, trajectory)

Le diagnostic peut désormais calculer les champs à la volée en appelant

In [ ]:
psi.diagnostic

Tous les champs spatiaux correspondant à l'évolution temporelle des variables du modèle sont renvoyés sous forme de tableau sur des points spécifiques $x=x_i$ et $y=y_i$ :

In [ ]:
psi.diagnostic.shape

On dispose ici de 200 001 pas de temps enregistrés pour un champ représenté par une grille 10 × 10. On peut tracer un seul champ au 100ème pas de temps enregistré avec :

In [ ]:
plt.imshow(psi.diagnostic[100], origin='lower')

ou utilisez la fonction de tracé de diagnostic intégrée :

In [ ]:
psi.plot(100)

qui interpole fortement les points de donnée $(x_i, y_i)$ pour obtenir la valeur en plus de points.

Pour conclure, veuillez noter que les coordonnées $x$ ou $y$ des points de la grille sont disponibles dans l'objet de diagnostic sous la forme :

In [ ]:
psi._X

et

In [ ]:
psi._Y

### Affichage des champs résultants sous forme d'animation

Grâce aux diagnostics, il est également possible de visualiser l'évolution temporelle des champs du modèle. Nous présentons ici simultanément un nuage de points des variables $\psi_{{\rm a}, 2}$ et $\psi_{{\rm a}, 3}$, ainsi que le champ de hauteur géopotentielle à 500 hPa en fonction de l'altitude orographique. Pour plus de détails sur les diagnostics et leur utilisation pour produire ce type d'animation, veuillez consulter cette [section](https://qgs.readthedocs.io/en/latest/files/user_guide.html#analyzing-the-model-s-output-with-diagnostics) du guide de l'utilisateur.

In [ ]:
psi = MiddleAtmosphericStreamfunctionDiagnostic(model_parameters, geopotential=True)
variable_nondim = VariablesDiagnostic([2, 1], model_parameters, False)
background = VariablesDiagnostic([2, 1], model_parameters, False)
background.set_data(time, trajectory)

m = MultiDiagnostic(1,2)
m.add_diagnostic(variable_nondim,
                 diagnostic_kwargs={'show_time': False, 'background': background},
                 plot_kwargs={'ms': 0.2})
m.add_diagnostic(psi,
                 diagnostic_kwargs={'style': 'contour', 'contour_labels': False},
                 plot_kwargs={'colors': 'k'})
# on introduit les données échantillonnées tous les 50 pas de temps
m.set_data(time[::10], trajectory[:, ::10])

m.animate(figsize=(20,6))

Cette évolution correspond à la propagation d'une onde de Rossby aux latitudes moyennes, un résultat obtenu dans les années 80 dans ce type de modèle barocline et montrant l'importance de l'instabilité barocline pour expliquer la variabilité atmosphérique à nos latitudes.

![ReadmeUrl](https://github.com/Climdyn/qgs/blob/master/misc/figs/readme.gif?raw=true "rossby ")

## Exercices

A faire par exemple en continuant ce notebook:

* Un premier exercice simple pour cette introduction est de redémarrer ce notebook en changeant des paramètres pour observer l'effet obtenu. Une description complète de tous les paramètres disponibles est fournie [ici](https://qgs.readthedocs.io/en/latest/files/technical/configuration.html).
* Intégrer les équations différentielles du modèle avec d'autres méthodes (par exemple la [méthode d'Euler](https://en.wikipedia.org/wiki/Euler_method) ou la [méthode de Heun](https://en.wikipedia.org/wiki/Heun's_method)), en les codant à la main à l'aide de la fonction `f` ($\boldsymbol{f}$) donnant les tendances du modèle, et comparer les trajectoires obtenues pour une même condition initiale.
* Changer le nombre de mode dans le modèle (voir la cellule [8] du notebook) pour obtenir le [modèle de Charney & Straus](https://journals.ametsoc.org/view/journals/atsc/37/6/1520-0469_1980_037_1157_fdimea_2_0_co_2.xml) vu au cours, et comparer la dynamique obtenue avec le modèle présenté ici. Le notebook suivant étudiera plus en détail ce modèle.
